
# ingest circuit csv file
- read file using spark dataframe reader api
- add metadata columns
 > source file,
 > ingestion timestamp
- wrrite to bronze delta table 

### read csv file using dataframe reader api

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get('p_batch_id')
v_batch_id

In [0]:
%run ../config/config-env


In [0]:
%run ../config/helper-function

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema =StructType([

    StructField('circuitId',    StringType()),
    StructField('url',          StringType()),
    StructField('circuitName',  StringType()),
    StructField('lat',          DoubleType()),
    StructField('long',         DoubleType()),
    StructField('locality',     StringType()),
    StructField('country',      StringType())
])


In [0]:
%python
circuits_df = (
                spark.read.format('csv')
                .option('header', True)
                .option('mode', 'FAILFAST')
                .schema(circuits_schema)
                .load(source_file)
               
               )

In [0]:
display(circuits_df)


###add metadat columns

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

In [0]:
circuits_final_df = circuits_final_df.withColumn('batch_id', F.lit(v_batch_id))


###write to bronze delta table


In [0]:

(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .partitionBy('batch_id')
        .option('replaceWhere', f"batch_id = '{v_batch_id}' ")
        .saveAsTable(table_name)
)

In [0]:
%sql
select * from formular1_incr.bronze.circuits;